In [83]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from gamePredModel import GamePredModel, mdn_loss, players_and_stats
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

In [138]:
INPUT_DIM = 1350
HIDDEN_DIM = 4096
OUTPUT_DIM = 150
N_COMPONENTS = 2
BATCH_SIZE = 64
GAME_ID = 22400367

In [139]:
data = pd.read_csv("../csv/data.csv")
all_cols = data.columns.tolist()
cols_to_drop = []
stats_to_drop =[
    'fieldgoalsattempted',
    'fieldgoalsmade',
    'fieldgoalspercentage',
    'threepointersattempted',
    'threepointersmade',
    'threepointerspercentage',
    'freethrowsattempted',
    'freethrowsmade',
    'freethrowspercentage',
    'reboundsdefensive',
    'reboundsoffensive',
    'reboundstotal',
    'foulspersonal',
]
for tn in ['t0', 't1']:
    for pidx in range(15):
        for stat in stats_to_drop:
            cols_to_drop.append(f'{tn}_p{pidx}_{stat}')

print(len(cols_to_drop), cols_to_drop)
data = data.drop(columns=cols_to_drop)



390 ['t0_p0_fieldgoalsattempted', 't0_p0_fieldgoalsmade', 't0_p0_fieldgoalspercentage', 't0_p0_threepointersattempted', 't0_p0_threepointersmade', 't0_p0_threepointerspercentage', 't0_p0_freethrowsattempted', 't0_p0_freethrowsmade', 't0_p0_freethrowspercentage', 't0_p0_reboundsdefensive', 't0_p0_reboundsoffensive', 't0_p0_reboundstotal', 't0_p0_foulspersonal', 't0_p1_fieldgoalsattempted', 't0_p1_fieldgoalsmade', 't0_p1_fieldgoalspercentage', 't0_p1_threepointersattempted', 't0_p1_threepointersmade', 't0_p1_threepointerspercentage', 't0_p1_freethrowsattempted', 't0_p1_freethrowsmade', 't0_p1_freethrowspercentage', 't0_p1_reboundsdefensive', 't0_p1_reboundsoffensive', 't0_p1_reboundstotal', 't0_p1_foulspersonal', 't0_p2_fieldgoalsattempted', 't0_p2_fieldgoalsmade', 't0_p2_fieldgoalspercentage', 't0_p2_threepointersattempted', 't0_p2_threepointersmade', 't0_p2_threepointerspercentage', 't0_p2_freethrowsattempted', 't0_p2_freethrowsmade', 't0_p2_freethrowspercentage', 't0_p2_reboundsdefens

In [140]:
print(data.shape)

(57340, 1534)


In [141]:
train_data = data.iloc[:, 34:]

In [142]:
train_data.head()

,t0_p0_points,t0_p1_points,t0_p2_points,t0_p3_points,t0_p4_points,t0_p5_points,t0_p6_points,t0_p7_points,t0_p8_points,t0_p9_points,...,t1_p5_numminutes,t1_p6_numminutes,t1_p7_numminutes,t1_p8_numminutes,t1_p9_numminutes,t1_p10_numminutes,t1_p11_numminutes,t1_p12_numminutes,t1_p13_numminutes,t1_p14_numminutes
0,21.0,11.0,7.0,8.0,7.0,0.0,0.0,0.0,0.0,0.0,...,13.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,33.0,28.0,18.0,6.0,0.0,6.0,16.0,5.0,0.0,0.0,...,18.0,16.0,12.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0
2,6.0,18.0,20.0,8.0,12.0,0.0,0.0,0.0,0.0,0.0,...,21.0,20.0,10.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0
3,35.0,17.0,24.0,14.0,12.0,6.0,0.0,0.0,0.0,0.0,...,16.0,11.0,9.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,14.0,15.0,19.0,4.0,10.0,11.0,6.0,0.0,0.0,0.0,...,14.0,12.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [143]:
# # Move first 30 columns to the end of the DataFrame
# first_30 = train_data.columns[:30].tolist()
# rest = train_data.columns[30:].tolist()
# train_data = train_data[rest + first_30]
# train_data

In [144]:
model = GamePredModel(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM, n_components=N_COMPONENTS)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = mdn_loss
num_epochs = 3

In [145]:




model = GamePredModel(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM, n_components=N_COMPONENTS)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = mdn_loss
num_epochs = 3

# Step 2: Split features and target
X = train_data.iloc[:, OUTPUT_DIM:]
y = train_data.iloc[:, :OUTPUT_DIM]

print(X.shape)
print(y.shape)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# # Step 4: To tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

# Step 5: DataLoader
train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True
)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=BATCH_SIZE)
print(train_data.shape)
train_data.describe()

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    epoch_start = time.time()

    for i, (batch_x, batch_y) in enumerate(train_loader, 1):
        batch_start = time.time()
        optimizer.zero_grad()

        # Forward pass
        pi, mu, sigma = model(batch_x)

        # Loss computation
        loss = mdn_loss(pi, mu, sigma, batch_y)
        total_loss += loss.item()

        # Backpropagation
        loss.backward()
        optimizer.step()

        batch_time = time.time() - batch_start
        if i % 100 == 0 or i == len(train_loader):
            print(
                f"Batch {i:3d}/{len(train_loader)} - Loss: {loss.item():.6f} - Time: {batch_time:.2f}s"
            )

    avg_train_loss = total_loss / len(train_loader)
    epoch_time = time.time() - epoch_start

    # ----- Evaluation -----
    model.eval()
    total_test_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            pi, mu, sigma = model(batch_x)
            test_loss = mdn_loss(pi, mu, sigma, batch_y)
            total_test_loss += test_loss.item()

    avg_test_loss = total_test_loss / len(test_loader)

    # ----- Logging -----
    print(f"\nEpoch [{epoch+1}/{num_epochs}] Summary:")
    print(f"  Train Loss : {avg_train_loss:.6f}")
    print(f"  Test Loss  : {avg_test_loss:.6f}")
    print(f"  Epoch Time : {epoch_time:.2f}s")
    print("-" * 60)


(57340, 1350)
(57340, 150)
(57340, 1500)
Batch 100/717 - Loss: 0.435147 - Time: 0.11s
Batch 200/717 - Loss: -0.095909 - Time: 0.11s
Batch 300/717 - Loss: 0.628951 - Time: 0.11s
Batch 400/717 - Loss: -0.250545 - Time: 0.11s
Batch 500/717 - Loss: -0.563578 - Time: 0.10s
Batch 600/717 - Loss: -0.281698 - Time: 0.11s
Batch 700/717 - Loss: 0.051593 - Time: 0.11s
Batch 717/717 - Loss: -0.105158 - Time: 0.10s

Epoch [1/3] Summary:
  Train Loss : 0.022023
  Test Loss  : -0.392616
  Epoch Time : 74.94s
------------------------------------------------------------
Batch 100/717 - Loss: -0.528553 - Time: 0.11s
Batch 200/717 - Loss: -0.397522 - Time: 0.11s
Batch 300/717 - Loss: -0.736391 - Time: 0.11s
Batch 400/717 - Loss: -0.269610 - Time: 0.11s
Batch 500/717 - Loss: -0.918027 - Time: 0.11s
Batch 600/717 - Loss: -0.511253 - Time: 0.11s
Batch 700/717 - Loss: -0.742390 - Time: 0.10s
Batch 717/717 - Loss: -0.852830 - Time: 0.10s

Epoch [2/3] Summary:
  Train Loss : -0.468447
  Test Loss  : -0.842045


In [146]:
# # After training
torch.save(model.state_dict(), "game_pred_mdn.pt")

In [147]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, "game_pred_mdn_full.pt")

In [148]:
model = GamePredModel(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM, n_components=N_COMPONENTS)
model.load_state_dict(torch.load("game_pred_mdn.pt"))
model.eval()  # set to evaluation mode

checkpoint = torch.load("game_pred_mdn_full.pt")
model.load_state_dict(checkpoint["model_state_dict"])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

In [149]:
game_data = data[data["game_id"] == GAME_ID].iloc[0, 34:]
X_input = game_data.iloc[OUTPUT_DIM:]
game_data

t0_p0_points          9.0
t0_p1_points         14.0
t0_p2_points         16.0
t0_p3_points         13.0
t0_p4_points          6.0
                     ... 
t1_p10_numminutes     0.0
t1_p11_numminutes     0.0
t1_p12_numminutes     0.0
t1_p13_numminutes     0.0
t1_p14_numminutes     0.0
Name: 12823, Length: 1500, dtype: object

In [150]:
X_input = X_input.astype(float)
X_input.describe()

count    1350.000000
mean        1.673128
std         8.347921
min         0.000000
25%         0.000000
50%         0.000000
75%         0.533250
max       248.000000
Name: 12823, dtype: float64

In [151]:
X_tensor = torch.tensor(X_input.values, dtype=torch.float32)
X_tensor = X_tensor.view(1, -1)
print(X_tensor)
model.eval()  # if not already in eval mode
with torch.no_grad():
    pi, mu, sigma = model(X_tensor)


tensor([[13.0000, 23.4000, 13.1000,  ...,  0.0000,  0.0000,  0.0000]])


In [152]:
pi

tensor([[[9.9982e-01, 1.8256e-04],
         [4.9256e-03, 9.9507e-01],
         [9.9804e-01, 1.9581e-03],
         [8.8381e-01, 1.1619e-01],
         [7.6474e-01, 2.3526e-01],
         [3.9204e-01, 6.0796e-01],
         [5.6108e-01, 4.3892e-01],
         [2.9167e-01, 7.0833e-01],
         [1.4711e-01, 8.5289e-01],
         [6.2462e-02, 9.3754e-01],
         [9.7663e-01, 2.3372e-02],
         [2.4572e-02, 9.7543e-01],
         [9.9113e-01, 8.8711e-03],
         [4.0842e-02, 9.5916e-01],
         [7.1131e-03, 9.9289e-01],
         [3.4864e-04, 9.9965e-01],
         [9.9446e-01, 5.5415e-03],
         [9.8066e-01, 1.9343e-02],
         [9.9155e-02, 9.0084e-01],
         [4.3151e-03, 9.9568e-01],
         [3.7540e-01, 6.2460e-01],
         [4.3841e-01, 5.6159e-01],
         [3.0286e-01, 6.9714e-01],
         [1.6553e-01, 8.3447e-01],
         [8.9690e-02, 9.1031e-01],
         [9.7960e-01, 2.0402e-02],
         [9.8453e-01, 1.5465e-02],
         [9.9107e-01, 8.9303e-03],
         [9.9343e-01

In [153]:
mu

tensor([[[ 1.8514e+01,  9.0212e-01],
         [ 3.4030e-02,  1.5458e+01],
         [ 1.2428e+01,  2.3700e-01],
         [ 1.0009e+01,  4.5194e-03],
         [ 8.2551e+00,  4.9719e-03],
         [-3.8694e-03,  6.7923e+00],
         [-5.3356e-04,  5.4914e+00],
         [ 4.2802e+00,  5.3442e-03],
         [ 3.7390e+00,  2.8282e-03],
         [ 2.8783e+00,  2.5162e-03],
         [-3.0817e-03,  1.3257e+00],
         [ 2.4359e-01, -1.4430e-03],
         [ 1.2015e-04,  8.5531e-02],
         [-2.0929e-01, -5.3066e-04],
         [ 1.4253e-02,  8.2225e-04],
         [ 6.2021e-01,  1.8632e+01],
         [ 1.5725e+01,  2.6867e-03],
         [ 1.3058e+01, -1.3674e-01],
         [ 1.9254e-02,  1.0544e+01],
         [-1.1180e-01,  7.3752e+00],
         [ 1.8833e-03,  7.0029e+00],
         [ 5.7287e+00, -7.6556e-04],
         [ 4.6667e+00, -2.8829e-03],
         [ 3.8518e+00, -6.8372e-03],
         [ 7.2061e-01,  1.9939e-03],
         [ 2.2899e-03,  1.7835e+00],
         [ 4.8336e-03,  2.1511e-01],
 

In [154]:
sigma

tensor([[[8.0229e+00, 9.6299e-01],
         [4.5377e-02, 7.4801e+00],
         [6.8625e+00, 2.4599e-02],
         [5.8585e+00, 2.4348e-02],
         [5.1200e+00, 1.9066e-02],
         [1.4414e-02, 4.4534e+00],
         [7.3694e-03, 3.7714e+00],
         [3.3498e+00, 1.0728e-02],
         [2.9087e+00, 7.1845e-03],
         [2.4733e+00, 6.3493e-03],
         [1.1250e-02, 1.4414e+00],
         [6.3019e-01, 1.4714e-02],
         [6.8038e-03, 5.1332e-01],
         [5.3516e-01, 1.7414e-02],
         [1.8202e-02, 9.8076e-03],
         [9.3135e-01, 7.8435e+00],
         [7.1425e+00, 4.8441e-02],
         [6.8961e+00, 3.1023e-02],
         [1.2889e-02, 6.0974e+00],
         [3.3172e-02, 5.7263e+00],
         [6.9585e-03, 4.6229e+00],
         [3.8838e+00, 8.9478e-03],
         [3.4042e+00, 8.7695e-03],
         [2.9659e+00, 1.1567e-02],
         [1.7918e+00, 9.5347e-03],
         [7.1653e-03, 2.0005e+00],
         [7.3563e-03, 2.8639e-01],
         [7.5145e-03, 7.5476e-01],
         [7.1783e-03

In [155]:
def sample_from_mdn(pi, mu, sigma):
    """
    pi:    (B, D, K) — mixture weights
    mu:    (B, D, K) — means
    sigma: (B, D, K) — stds

    Returns:
        samples: (B, D)
    """
    B, D, K = pi.shape

    # Step 1: Sample a component index from categorical distribution for each output dim
    categorical = torch.distributions.Categorical(pi)
    component_indices = categorical.sample()  # (B, D) — mixture component chosen per output

    # Step 2: Gather mu and sigma corresponding to sampled component
    batch_indices = torch.arange(B).unsqueeze(1).expand(B, D)  # (B, D)
    dim_indices = torch.arange(D).unsqueeze(0).expand(B, D)    # (B, D)

    # Gather the corresponding mu and sigma based on sampled component
    chosen_mu = mu[batch_indices, dim_indices, component_indices]
    chosen_sigma = sigma[batch_indices, dim_indices, component_indices]

    # Step 3: Sample from normal distribution
    normal = torch.distributions.Normal(chosen_mu, chosen_sigma)
    samples = normal.sample()  # (B, D)
    samples = torch.clamp(samples, min=0.0)

    return samples

In [156]:
samples = sample_from_mdn(pi, mu, sigma)  # Shape: (1, 360)

In [157]:
stats_df = players_and_stats(samples, GAME_ID, data)

     personId     points   assists    blocks    steals  rebounds
0   1628384.0   8.246575  6.225491  0.697726  0.238667  0.464307
1   1628973.0  12.079942  1.258709  0.002484  0.445299  0.830825
2   1630540.0  16.092461  4.199175  0.014264  1.182621  0.712251
3   1630173.0  10.541114  3.491971  0.000000  0.000000  0.043093
4   1630579.0   6.534563  4.256457  2.039735  0.001200  1.063739
5   1631210.0   9.806719  0.017984  0.009945  1.794422  1.662439
6         0.0   0.004320  1.307955  0.015809  0.000000  0.000000
7         0.0   0.006954  0.000000  0.012273  0.215655  0.007848
8         0.0   0.000000  0.000000  0.006588  0.007017  0.000000
9         0.0   0.001751  0.115898  0.004808  0.000000  0.015054
10        0.0   0.016154  0.000000  0.000000  0.005504  0.000000
11        0.0   0.011092  0.009601  0.004044  0.001528  0.000146
12        0.0   0.000000  0.000000  0.274719  0.016335  0.001718
13        0.0   0.000995  0.000000  0.001929  0.000000  0.007749
14        0.0   0.008490 

In [158]:
print(h["points"].sum(), a["points"].sum())

NameError: name 'h' is not defined

In [ ]:
h

firstName,lastName,personId,points,assists,blocks,steals,reboundsTotal,turnovers
str,str,i64,f32,f32,f32,f32,f32,f32
"""Anthony""","""Edwards""",1630162,25.803543,9.661681,0.0,0.947284,9.076491,1.751937
"""Julius""","""Randle""",203944,39.267982,3.921318,0.224781,0.830468,9.787584,0.767914
"""Donte""","""DiVincenzo""",1628978,10.460669,2.587386,0.0,0.792567,2.908532,0.939958
"""Jaden""","""McDaniels""",1630183,6.931751,0.0,0.010324,0.0,11.6703,2.813545
"""Rudy""","""Gobert""",203497,11.035434,1.689722,0.003017,2.051392,4.685094,4.405427
…,…,…,…,…,…,…,…,…
"""Luka""","""Garza""",1630568,0.0,0.009612,0.0,0.0,0.014725,0.0
"""Josh""","""Minott""",1631169,0.0,0.0,0.0,0.007492,0.0,0.0
"""PJ""","""Dozier""",1628408,0.003239,0.019572,0.0,0.0,0.0,0.004313


In [ ]:
a

firstName,lastName,personId,points,assists,blocks,steals,reboundsTotal,turnovers
str,str,i64,f32,f32,f32,f32,f32,f32
"""Karl-Anthony""","""Towns""",1626157,24.767124,13.377798,0.02243,0.0,3.038804,0.856735
"""Mikal""","""Bridges""",1628969,14.453997,5.02913,2.606043,0.728308,10.440793,9.481155
"""OG""","""Anunoby""",1628384,27.45118,12.225641,0.00204,2.314866,10.908648,1.49482
"""Jalen""","""Brunson""",1628973,24.209377,1.143102,1.022304,0.003147,8.669761,1.406437
"""Miles""","""McBride""",1630540,8.667492,11.1581,0.001319,0.577933,14.273738,1.795496
…,…,…,…,…,…,…,…,…
"""Ariel""","""Hukporti""",1630574,0.0,0.005204,0.0,0.013652,0.003178,0.0
"""Tyler""","""Kolek""",1642278,0.000803,0.0,0.0,0.001046,0.0,0.0
"""Jacob""","""Toppin""",1631210,0.003803,0.0,0.003229,0.0,0.00247,0.008193
